# 06 — Knowledge Freshness & Cache Invalidation Demo

Companion notebook to `06-knowledge-freshness-and-conversation-state-lifecycle.md` — the chapter
that directly answers the likely follow-up question: *"how do you handle it when the source
documents behind this chatbot's answers get updated or corrected?"*

It implements, end to end and offline, the exact logic described in that chapter's prose:

1. A mocked Azure AI Search index whose chunk IDs are derived from `document_id`, with **no
   version/supersede concept** — matching course 05's confirmed finding that every upload, including
   a revision, gets a brand-new `document_id` with no linkage to what it replaces.
2. The **manual workaround** (chapter 06, Part 2): search, then delete-by-filter on the stale
   `document_id`, using only a capability Azure AI Search genuinely has today.
3. Two caches with deliberately different invalidation strategies (chapter 06, Part 4): a
   **content-addressed embedding cache** (fails safe by construction) and a **citation-indexed
   full-answer cache** (needs an explicit purge hook).
4. The **proposed event-driven fix**: a `document_superseded` event that reaches both the index and
   the answer cache in a single step, instead of a person having to remember both.
5. A concrete, runnable proof that **conversation turn count and knowledge freshness are orthogonal**
   (chapter 06, Part 3) — a brand-new, zero-turn session and a forty-turn-deep session get exactly
   the same (stale) answer, because the staleness lives in the index, not in anything session-scoped.

Everything here is a self-contained, offline mock — no real Azure AI Search, no real embedding model,
no network calls — matching this course's "runs offline" rule for notebooks. As with the rest of this
course, this is an **illustrative, plausible reconstruction** of what a system like this would do, not
a verified description of a real deployed system.

## 1. A mocked Azure AI Search index — no version concept, exactly like today's real gap

`MockSearchIndex` stands in for Azure AI Search. Chunk IDs are `f"{document_id}::chunk{i}"` — derived
purely from `document_id` and position, with nothing linking two documents that happen to be revisions
of the same logical policy. This mirrors chapter 06 Part 1 exactly: since course 05 confirmed every
upload gets a brand-new `document_id` with no `supersedes_document_id` link, an index keyed this way has
no way to know two sets of chunks are related at all.

In [1]:
import hashlib
import time
import uuid
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set


@dataclass
class Chunk:
    chunk_id: str
    document_id: str
    document_title: str
    business_line: str
    text: str
    indexed_as_of: float


class MockSearchIndex:
    """
    Offline stand-in for Azure AI Search. Chunk IDs are derived from
    document_id + chunk position -- NOT from a stable logical-document
    identity -- exactly the "today" behavior described in chapter 06 Part 1.
    No supersede/version awareness anywhere.
    """

    def __init__(self):
        self._chunks: Dict[str, Chunk] = {}

    def index_document(self, document_id: str, title: str, business_line: str, chunks: List[str]) -> List[str]:
        chunk_ids = []
        for i, text in enumerate(chunks):
            chunk_id = f"{document_id}::chunk{i}"
            self._chunks[chunk_id] = Chunk(
                chunk_id=chunk_id,
                document_id=document_id,
                document_title=title,
                business_line=business_line,
                text=text,
                indexed_as_of=time.time(),
            )
            chunk_ids.append(chunk_id)
        return chunk_ids

    def search(self, query_terms: List[str], business_line: Optional[str] = None) -> List[Chunk]:
        """Extremely naive keyword search standing in for hybrid vector+BM25 search --
        good enough to demonstrate the staleness problem without needing a real embedding model."""
        results = []
        for chunk in self._chunks.values():
            if business_line is not None and chunk.business_line != business_line:
                continue
            if any(term.lower() in chunk.text.lower() for term in query_terms):
                results.append(chunk)
        return results

    def delete_by_document_id(self, document_id: str) -> int:
        """Mirrors Azure AI Search's delete-by-filter capability: `document_id eq '<id>'`."""
        to_delete = [cid for cid, c in self._chunks.items() if c.document_id == document_id]
        for cid in to_delete:
            del self._chunks[cid]
        return len(to_delete)

    def all_chunks_for_title(self, title: str) -> List[Chunk]:
        return [c for c in self._chunks.values() if c.document_title == title]


index = MockSearchIndex()
print("MockSearchIndex ready -- no version concept, exactly like the real gap chapter 06 describes.")

MockSearchIndex ready -- no version concept, exactly like the real gap chapter 06 describes.


## 2. Proving the staleness problem: a revision creates a parallel, unlinked chunk set

This reproduces exactly what chapter 06 Part 1 describes: a policy correction (refund window 5–7 days
→ 3–5 days) gets uploaded as a brand-new `document_id`, per course 05's confirmed upload-path behavior.
Both the old and new chunks end up equally discoverable, with no signal about which is authoritative.

In [2]:
# Upload v1 of a refund policy (mirrors course 05's upload -> brand-new document_id every time)
v1_id = str(uuid.uuid4())
index.index_document(
    v1_id, "Refund_Policy", "GENERAL",
    ["Refunds are processed within 5-7 business days of the request being approved."],
)

# Later, a correction ships: refunds now take 3-5 business days. Per course 05's confirmed
# upload-path behavior, this gets an entirely new document_id -- there is no supersede link.
v2_id = str(uuid.uuid4())
index.index_document(
    v2_id, "Refund_Policy", "GENERAL",
    ["Refunds are processed within 3-5 business days of the request being approved."],
)

results = index.search(["refund"], business_line="GENERAL")
print(f"Search for 'refund' returns {len(results)} chunk(s):")
for r in results:
    print(f"  - document_id={r.document_id[:8]}...  text={r.text!r}")

assert len(results) == 2, "Both the stale v1 chunk and the current v2 chunk are discoverable -- no linkage."
print()
print("Confirmed: exactly like course 05's finding for the upload side, the retrieval index has")
print("no idea these two documents are related -- both are equally 'findable', with no ranking")
print("signal distinguishing current from superseded (chapter 06, Part 1).")

Search for 'refund' returns 2 chunk(s):
  - document_id=680a0799...  text='Refunds are processed within 5-7 business days of the request being approved.'
  - document_id=84a3689e...  text='Refunds are processed within 3-5 business days of the request being approved.'

Confirmed: exactly like course 05's finding for the upload side, the retrieval index has
no idea these two documents are related -- both are equally 'findable', with no ranking
signal distinguishing current from superseded (chapter 06, Part 1).


## 3. The manual workaround (chapter 06, Part 2) — delete-by-filter, entirely operator-driven

Nothing here requires a new platform capability — `delete_by_document_id` mirrors Azure AI Search's
real delete-by-filter operation (`document_id eq '<stale-id>'`). The gap is that nothing calls it
automatically.

In [3]:
deleted_count = index.delete_by_document_id(v1_id)
print(f"Manually deleted {deleted_count} stale chunk(s) for document_id={v1_id[:8]}...")

results_after = index.search(["refund"], business_line="GENERAL")
print(f"Search for 'refund' now returns {len(results_after)} chunk(s):")
for r in results_after:
    print(f"  - document_id={r.document_id[:8]}...  text={r.text!r}")

assert len(results_after) == 1
assert results_after[0].document_id == v2_id
print()
print("Confirmed: the manual workaround (search, then delete-by-filter) leaves exactly one")
print("current chunk discoverable -- the pieces exist and work today, they're just not automated.")

Manually deleted 1 stale chunk(s) for document_id=680a0799...
Search for 'refund' now returns 1 chunk(s):
  - document_id=84a3689e...  text='Refunds are processed within 3-5 business days of the request being approved.'

Confirmed: the manual workaround (search, then delete-by-filter) leaves exactly one
current chunk discoverable -- the pieces exist and work today, they're just not automated.


## 4. Two caches, two invalidation strategies (chapter 06, Part 4)

- **`EmbeddingCache`** is keyed on `sha256(chunk_text) + embedding_model_version` — content-addressed,
  so a changed chunk naturally gets a new key. It **fails safe by construction**: the old entry doesn't
  actively serve wrong data, it just becomes unreferenced.
- **`AnswerCache`** is keyed on normalized question text with a TTL, plus an explicit **citation
  index** (`document_id -> set of cache keys`) so a `document_superseded` event can purge precisely the
  entries that cited the now-stale document — this is the reverse index chapter 06 describes as the
  mechanism behind step 2(c) of the proposed worker.

In [4]:
class EmbeddingCache:
    """Content-addressed: key = sha256(chunk_text) + embedding_model_version.
    A changed chunk naturally gets a new key -- the old entry just goes unreferenced
    rather than actively serving wrong data. This is the 'fail safe by default' design
    from chapter 06, Part 4."""

    def __init__(self, model_version: str):
        self.model_version = model_version
        self._store: Dict[str, List[float]] = {}
        self.hits = 0
        self.misses = 0

    def _key(self, text: str) -> str:
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
        return f"emb:{digest}:{self.model_version}"

    def get_or_embed(self, text: str) -> List[float]:
        key = self._key(text)
        if key in self._store:
            self.hits += 1
            return self._store[key]
        self.misses += 1
        # mock embedding: deterministic, content-derived vector so equal text -> equal vector
        vec = [float(b) for b in hashlib.sha256(text.encode("utf-8")).digest()[:4]]
        self._store[key] = vec
        return vec


class AnswerCache:
    """Keyed on normalized question text, short TTL, with an explicit citation index
    (document_id -> set of cache keys) so an event can purge precisely the entries that
    cited a now-superseded document -- chapter 06, Part 4, item 2(c)."""

    def __init__(self, ttl_seconds: float):
        self.ttl_seconds = ttl_seconds
        self._store: Dict[str, tuple] = {}   # key -> (answer, cached_at, cited_document_ids)
        self._citation_index: Dict[str, Set[str]] = {}  # document_id -> set of cache keys

    @staticmethod
    def _normalize(question: str) -> str:
        return question.strip().lower()

    def put(self, question: str, answer: str, cited_document_ids: List[str]) -> None:
        key = self._normalize(question)
        self._store[key] = (answer, time.time(), set(cited_document_ids))
        for doc_id in cited_document_ids:
            self._citation_index.setdefault(doc_id, set()).add(key)

    def get(self, question: str) -> Optional[str]:
        key = self._normalize(question)
        entry = self._store.get(key)
        if entry is None:
            return None
        answer, cached_at, _ = entry
        if time.time() - cached_at > self.ttl_seconds:
            del self._store[key]
            return None
        return answer

    def evict_by_document_id(self, document_id: str) -> int:
        """The event-driven purge from chapter 06, Part 4, item 2(c)."""
        keys = self._citation_index.pop(document_id, set())
        evicted = 0
        for key in keys:
            if key in self._store:
                del self._store[key]
                evicted += 1
        return evicted


emb_cache = EmbeddingCache(model_version="text-embedding-3-small-v1")
answer_cache = AnswerCache(ttl_seconds=600)  # illustrative: minutes, not hours, per chapter 06/07

# Populate the answer cache with an answer grounded in the (now-stale) v1 chunk
question = "How long do refunds take?"
answer_cache.put(question, "Refunds are processed within 5-7 business days.", cited_document_ids=[v1_id])

cached = answer_cache.get(question)
print(f"Answer cache hit before invalidation: {cached!r}")
assert cached is not None

Answer cache hit before invalidation: 'Refunds are processed within 5-7 business days.'


## 5. The proposed event-driven fix (chapter 06, Part 4) — one event, two systems updated together

`on_document_superseded` is a faithful implementation of chapter 06's proposed index-sync worker: a
single `document_superseded` event reaches **both** the search index and the answer cache, rather than
relying on an operator to remember both steps of the manual workaround.

In [5]:
def on_document_superseded(event: dict, index: MockSearchIndex, answer_cache: AnswerCache) -> None:
    """Faithful implementation of chapter 06's proposed index-sync worker logic."""
    old_id = event["old_document_id"]
    deleted = index.delete_by_document_id(old_id)
    evicted = answer_cache.evict_by_document_id(old_id)
    print(f"  [event] document_superseded: old={old_id[:8]}... new={event['new_document_id'][:8]}...")
    print(f"  [event] deleted {deleted} index chunk(s), evicted {evicted} cached answer(s)")


print("Firing a document_superseded event (v1 -> v2) through the proposed worker...")
event = {
    "event": "document_superseded",
    "old_document_id": v1_id,
    "new_document_id": v2_id,
}
on_document_superseded(event, index, answer_cache)

cached_after = answer_cache.get(question)
print(f"Answer cache hit after invalidation: {cached_after!r}")
assert cached_after is None, "The stale cached answer should be gone after the supersede event."
print("Confirmed: the event reached both the index (sections 2/3) and the answer cache in one step.")

Firing a document_superseded event (v1 -> v2) through the proposed worker...
  [event] document_superseded: old=680a0799... new=84a3689e...
  [event] deleted 0 index chunk(s), evicted 1 cached answer(s)
Answer cache hit after invalidation: None
Confirmed: the event reached both the index (sections 2/3) and the answer cache in one step.


## 6. Conversation turn count vs. knowledge freshness — proving they're orthogonal (chapter 06, Part 3)

We reintroduce a stale duplicate chunk (undoing section 3's cleanup, purely for this demo) and ask the
same question from two sessions: a brand-new, zero-prior-turn session, and a session forty turns deep.
If knowledge freshness were somehow tied to conversation length, the two sessions would behave
differently. They don't — which is exactly the point.

In [6]:
@dataclass
class ConversationSession:
    session_id: str
    turns: List[str] = field(default_factory=list)


def ask(index: MockSearchIndex, session: ConversationSession, question: str, terms: List[str]) -> str:
    session.turns.append(question)
    hits = index.search(terms, business_line="GENERAL")
    if not hits:
        return "I don't know."
    # naive "grounding": surface the most-recently-indexed matching chunk's text
    hits.sort(key=lambda c: c.indexed_as_of, reverse=True)
    return hits[0].text


# Re-introduce a stale duplicate to demonstrate the point (undo section 3's cleanup for this demo)
stale_id = str(uuid.uuid4())
index.index_document(stale_id, "Refund_Policy", "GENERAL", ["Refunds are processed within 5-7 business days."])

fresh_session = ConversationSession(session_id="brand-new-session")
fresh_answer = ask(index, fresh_session, "How long do refunds take?", ["refund"])
print(f"Fresh, zero-prior-turn session asks about refunds -> {fresh_answer!r}")
print(f"Turns so far in this session: {len(fresh_session.turns)}")

long_session = ConversationSession(session_id="forty-turns-in")
for i in range(40):
    long_session.turns.append(f"filler turn {i}")
long_answer = ask(index, long_session, "How long do refunds take?", ["refund"])
print(f"\n40-turns-deep session asks the same question -> {long_answer!r}")

assert fresh_answer == long_answer, "Turn count should have zero effect on which chunk is retrieved."
print()
print("Both sessions see whichever chunk the index happens to rank first -- turn count (0 vs 40)")
print("had zero effect on whether the answer is grounded in current or stale content. That's the")
print("concrete proof that conversation-memory management and knowledge freshness are orthogonal")
print("axes (chapter 06, Part 3): clearing history would not have changed either answer above.")

# Clean up the stale duplicate the "real" way this demo cares about, to leave the index consistent
index.delete_by_document_id(stale_id)
final_results = index.search(["refund"], business_line="GENERAL")
assert len(final_results) == 1 and final_results[0].document_id == v2_id
print("\nIndex left in a clean, single-current-version state at the end of the notebook.")

Fresh, zero-prior-turn session asks about refunds -> 'Refunds are processed within 5-7 business days.'
Turns so far in this session: 1

40-turns-deep session asks the same question -> 'Refunds are processed within 5-7 business days.'

Both sessions see whichever chunk the index happens to rank first -- turn count (0 vs 40)
had zero effect on whether the answer is grounded in current or stale content. That's the
concrete proof that conversation-memory management and knowledge freshness are orthogonal
axes (chapter 06, Part 3): clearing history would not have changed either answer above.

Index left in a clean, single-current-version state at the end of the notebook.


## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1–2 | A mock index with no version concept; a revision creates a parallel, unlinked chunk set | Chapter 06, Part 1 — today's real gap, given course 05's confirmed no-versioning upload behavior |
| 3 | The manual search-then-delete-by-filter workaround, running end to end | Chapter 06, Part 2 — achievable today with existing Azure AI Search capability |
| 4 | Two cache designs with different invalidation strategies (content-addressed vs. citation-indexed) | Chapter 06, Part 4 — the proposed cache design |
| 5 | One `document_superseded` event invalidating both the index and the answer cache | Chapter 06, Part 4 — the proposed event-driven worker |
| 6 | A zero-turn session and a forty-turn session get the identical (stale) answer | Chapter 06, Part 3 — conversation turn count and knowledge freshness are orthogonal axes |

Section 6 is the notebook's sharpest point: nothing about conversation length protects against, or
causes, a knowledge-freshness problem — the fix has to happen at the index/cache layer (sections 3–5),
not the conversation-memory layer (Chapter 3 / `03_simple_chatbot_with_memory.ipynb`).